## install

In [ ]:
reqs = """sentence-transformers>=4.0,<6
transformers>=4.45,<5
torch>=2.5
datasets>=2.19
pandas
pyarrow
numpy<2
scikit-learn
tqdm
huggingface_hub
"""
with open("requirements.txt", "w") as f:
    f.write(reqs)
print(open("requirements.txt").read())

Cell 2 — imports and device check

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Cell 3 — pull the ESCI parquet files

In [ ]:
import subprocess, pathlib

ESCI_DIR = pathlib.Path("esci-data")
if not ESCI_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/amazon-science/esci-data.git"],
        check=True,
    )
    subprocess.run(["git", "lfs", "pull"], cwd=str(ESCI_DIR), check=True)

DATA = ESCI_DIR / "shopping_queries_dataset"
print(sorted(p.name for p in DATA.glob("*")))

Cell 4 — load, merge, filter to English test split

In [ ]:
df_examples = pd.read_parquet(DATA / "shopping_queries_dataset_examples.parquet")
df_products = pd.read_parquet(DATA / "shopping_queries_dataset_products.parquet")

df = pd.merge(
    df_examples,
    df_products,
    how="left",
    on=["product_locale", "product_id"],
)

df = df[df["small_version"] == 1]
df = df[df["split"] == "test"]
df = df[df["product_locale"] == "us"].copy()

print("rows:", len(df))
print("unique queries:", df["query_id"].nunique())
print(df["esci_label"].value_counts(normalize=True).round(3))

Cell 5 — graded relevance mapping + product-text recipe

In [ ]:
label2gain = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
df["gain"] = df["esci_label"].map(label2gain)

def build_product_text(row):
    return str(row["product_title"]) if pd.notna(row["product_title"]) else ""

df["product_text"] = df.apply(build_product_text, axis=1)

df = df[df["product_text"].str.len() > 0].copy()
print("rows after dropping empty product_text:", len(df))
df[["query", "product_text", "esci_label", "gain"]].head(8)

Cell 6 — load the off-the-shelf baseline reranker

In [ ]:
from sentence_transformers import CrossEncoder

baseline = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
    device=device,
)
print("baseline loaded")

Cell 7 — score every (query, product_text) pair

In [ ]:
pairs = list(zip(df["query"].tolist(), df["product_text"].tolist()))

scores = baseline.predict(
    pairs,
    batch_size=64,
    show_progress_bar=True,
)
df["baseline_score"] = scores
print("scored:", len(scores))

Cell 8 — nDCG@k implementation

In [ ]:
def dcg_at_k(gains, k):
    gains = np.asarray(gains, dtype=float)[:k]
    if gains.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains / discounts))

def ndcg_at_k(true_gains, scores, k):
    order = np.argsort(-np.asarray(scores))
    ranked_gains = np.asarray(true_gains)[order]
    dcg = dcg_at_k(ranked_gains, k)
    ideal_gains = np.sort(true_gains)[::-1]
    idcg = dcg_at_k(ideal_gains, k)
    return dcg / idcg if idcg > 0 else 0.0

def mean_ndcg(frame, score_col, k):
    vals = []
    for _, g in frame.groupby("query_id"):
        if len(g) < 2:
            continue
        vals.append(ndcg_at_k(g["gain"].values, g[score_col].values, k))
    return float(np.mean(vals)), len(vals)

Cell 9 — the "before" number

In [ ]:
for k in [5, 10, 20]:
    score, n = mean_ndcg(df, "baseline_score", k)
    print(f"baseline nDCG@{k}: {score:.4f}  (over {n} queries)")

Cell 10 — load the train split

In [ ]:
df_train = pd.merge(
    df_examples,
    df_products,
    how="left",
    on=["product_locale", "product_id"],
)
df_train = df_train[df_train["small_version"] == 1]
df_train = df_train[df_train["split"] == "train"]
df_train = df_train[df_train["product_locale"] == "us"].copy()

df_train["gain"] = df_train["esci_label"].map(label2gain)
df_train["product_text"] = df_train.apply(build_product_text, axis=1)
df_train = df_train[df_train["product_text"].str.len() > 0].copy()

print("train rows:", len(df_train))
print("train unique queries:", df_train["query_id"].nunique())
print(df_train["esci_label"].value_counts(normalize=True).round(3))

Cell 11 — carve a validation set out of train (by query, no leakage)

In [ ]:
rng = np.random.default_rng(42)
train_qids = df_train["query_id"].unique()
rng.shuffle(train_qids)

n_val = int(0.05 * len(train_qids))
val_qids = set(train_qids[:n_val])

df_val = df_train[df_train["query_id"].isin(val_qids)].copy()
df_fit = df_train[~df_train["query_id"].isin(val_qids)].copy()

print("fit rows:", len(df_fit), "| fit queries:", df_fit["query_id"].nunique())
print("val rows:", len(df_val), "| val queries:", df_val["query_id"].nunique())
assert len(val_qids & set(df_fit["query_id"])) == 0

Cell 12 — build HuggingFace Datasets for the trainer

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_dict({
    "query": df_fit["query"].astype(str).tolist(),
    "product_text": df_fit["product_text"].astype(str).tolist(),
    "label": df_fit["gain"].astype(float).tolist(),
})

eval_ds = Dataset.from_dict({
    "query": df_val["query"].astype(str).tolist(),
    "product_text": df_val["product_text"].astype(str).tolist(),
    "label": df_val["gain"].astype(float).tolist(),
})

print(train_ds)
print(eval_ds)

Cell 13 — fresh model instance to fine-tune, and the loss

In [ ]:
import torch.nn as nn
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=256,
    num_labels=1,
    device=device,
)

loss = MSELoss(model, activation_fn=nn.Sigmoid())
print("model + loss ready")

Cell 14 — wrap our graded nDCG as a custom evaluator the trainer can call

In [ ]:
from sentence_transformers.evaluation import SentenceEvaluator

class GradedNDCGEvaluator(SentenceEvaluator):
    def __init__(self, frame, k=10, batch_size=64, name="val"):
        self.frame = frame
        self.k = k
        self.batch_size = batch_size
        self.name = name
        self.primary_metric = f"{name}_ndcg@{k}"
        self.pairs = list(zip(frame["query"].tolist(), frame["product_text"].tolist()))

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        scores = model.predict(self.pairs, batch_size=self.batch_size, show_progress_bar=False)
        tmp = self.frame[["query_id", "gain"]].copy()
        tmp["s"] = scores
        vals = []
        for _, g in tmp.groupby("query_id"):
            if len(g) < 2:
                continue
            vals.append(ndcg_at_k(g["gain"].values, g["s"].values, self.k))
        score = float(np.mean(vals))
        return {self.primary_metric: score}

val_evaluator = GradedNDCGEvaluator(df_val, k=10, name="val")

Cell 15 — training arguments tuned

In [ ]:
args = CrossEncoderTrainingArguments(
    output_dir="cartographer-run",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ndcg@10",
    greater_is_better=True,
    dataloader_num_workers=2,
    seed=42,
)

Cell 16 — train

In [ ]:
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss,
    evaluator=val_evaluator,
)
trainer.train()

Cell 17 — score the sealed test set with the fine-tuned model

In [ ]:
test_pairs = list(zip(df["query"].tolist(), df["product_text"].tolist()))

In [ ]:
test_pairs = list(zip(df["query"].tolist(), df["product_text"].tolist()))

ft_scores = model.predict(
    test_pairs,
    batch_size=64,
    show_progress_bar=True,
)
df["ft_score"] = ft_scores
print("scored:", len(ft_scores))

Cell 18 — the before/after table, the headline artifact

In [ ]:
print(f"{'metric':<14}{'baseline':>12}{'cartographer':>15}{'Δ abs':>10}{'Δ rel':>10}")
print("-" * 61)
for k in [5, 10, 20]:
    base, n = mean_ndcg(df, "baseline_score", k)
    ft, _ = mean_ndcg(df, "ft_score", k)
    d_abs = ft - base
    d_rel = 100 * d_abs / base
    print(f"nDCG@{k:<9}{base:>12.4f}{ft:>15.4f}{d_abs:>+10.4f}{d_rel:>+9.2f}%")

Cell 19 — MAP and MRR, the secondary metrics §7 asks for

In [ ]:
def ap_at_k(true_gains, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    rel = (np.asarray(true_gains)[order] > 0).astype(float)
    if rel.sum() == 0:
        return 0.0
    precisions = np.cumsum(rel) / (np.arange(len(rel)) + 1)
    return float((precisions * rel).sum() / rel.sum())

def rr_at_k(true_gains, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    rel = (np.asarray(true_gains)[order] > 0).astype(float)
    hit = np.nonzero(rel)[0]
    return float(1.0 / (hit[0] + 1)) if hit.size else 0.0

def mean_metric(frame, score_col, fn, k):
    vals = []
    for _, g in frame.groupby("query_id"):
        if len(g) < 2:
            continue
        vals.append(fn(g["gain"].values, g[score_col].values, k))
    return float(np.mean(vals))

for name, fn in [("MAP@10", ap_at_k), ("MRR@10", rr_at_k)]:
    base = mean_metric(df, "baseline_score", fn, 10)
    ft = mean_metric(df, "ft_score", fn, 10)
    print(f"{name}: {base:.4f} -> {ft:.4f}  ({100*(ft-base)/base:+.2f}%)")

Cell 20 — the max_length fairness check (kill the asymmetry objection first)

In [ ]:
from sentence_transformers import CrossEncoder

baseline_256 = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=256,
    device=device,
)
b256 = baseline_256.predict(test_pairs, batch_size=64, show_progress_bar=True)
df["baseline_256_score"] = b256

for k in [5, 10, 20]:
    a, _ = mean_ndcg(df, "baseline_score", k)
    b, _ = mean_ndcg(df, "baseline_256_score", k)
    print(f"nDCG@{k}: 512={a:.4f}  256={b:.4f}  diff={b-a:+.4f}")

Cell 21 — per-label score separation (does the model actually order E>S>C>I?)

In [ ]:
for col, tag in [("baseline_score", "baseline"), ("ft_score", "cartographer")]:
    print(f"\n{tag} — mean score by true label:")
    g = df.groupby("esci_label")[col].agg(["mean", "std", "count"])
    print(g.reindex(["E", "S", "C", "I"]).round(4))

Cell 22 — epoch-2 check with EarlyStopping wired in

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss
from transformers import EarlyStoppingCallback
import torch.nn as nn

model_e2 = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=256,
    num_labels=1,
    device=device,
)
loss_e2 = MSELoss(model_e2, activation_fn=nn.Sigmoid())
val_evaluator_e2 = GradedNDCGEvaluator(df_val, k=10, name="val")

args_e2 = CrossEncoderTrainingArguments(
    output_dir="cartographer-run-e2",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ndcg@10",
    greater_is_better=True,
    dataloader_num_workers=2,
    seed=42,
)

trainer_e2 = CrossEncoderTrainer(
    model=model_e2,
    args=args_e2,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss_e2,
    evaluator=val_evaluator_e2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
trainer_e2.train()

 score the epoch-2 model on sealed test, compare to epoch-1

In [ ]:
import os
for d in ["cartographer-run", "cartographer-run-e2"]:
    print(d, ":")
    if os.path.exists(d):
        for x in sorted(os.listdir(d)):
            print("   ", x)
    else:
        print("    (missing)")

In [ ]:
from sentence_transformers import CrossEncoder
baseline_256 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=256, device=device)
test_pairs = list(zip(df["query"].tolist(), df["product_text"].tolist()))
df["baseline_256_score"] = baseline_256.predict(test_pairs, batch_size=64, show_progress_bar=True)

In [ ]:
model    = CrossEncoder("cartographer-run/checkpoint-5000",    max_length=256, device=device)
model_e2 = CrossEncoder("cartographer-run-e2/checkpoint-9500", max_length=256, device=device)
print("both models loaded")
df["ft_score"]    = model.predict(test_pairs, batch_size=64, show_progress_bar=True)
df["ft_e2_score"] = model_e2.predict(test_pairs, batch_size=64, show_progress_bar=True)

print(f"{'metric':<12}{'epoch1':>10}{'epoch2':>10}{'Δ':>10}")
print("-" * 42)
for k in [5, 10, 20]:
    e1, _ = mean_ndcg(df, "ft_score", k)
    e2, _ = mean_ndcg(df, "ft_e2_score", k)
    print(f"nDCG@{k:<7}{e1:>10.4f}{e2:>10.4f}{e2-e1:>+10.4f}")

In [ ]:
model_e2.save_pretrained("cartographer-best")
print("saved to cartographer-best/")

In [ ]:
from sentence_transformers import CrossEncoder
_check = CrossEncoder("cartographer-best", max_length=256, device=device)
_s = _check.predict(test_pairs, batch_size=64, show_progress_bar=True)
df["_check_score"] = _s
v, _ = mean_ndcg(df, "_check_score", 10)
print(f"reloaded nDCG@10: {v:.4f}  (expect 0.7642)")

Cell — per-label score behavior on the epoch-2

In [ ]:
from sentence_transformers import CrossEncoder

test_pairs = list(zip(df["query"].tolist(), df["product_text"].tolist()))

baseline_256 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=256, device=device)
df["baseline_256_score"] = baseline_256.predict(test_pairs, batch_size=64, show_progress_bar=True)

model_e2 = CrossEncoder("cartographer-best", max_length=256, device=device)
df["ft_e2_score"] = model_e2.predict(test_pairs, batch_size=64, show_progress_bar=True)

print("\ncolumns now:", [c for c in df.columns if "score" in c])
for k in [5, 10, 20]:
    b, _ = mean_ndcg(df, "baseline_256_score", k)
    f, _ = mean_ndcg(df, "ft_e2_score", k)
    print(f"nDCG@{k}: baseline={b:.4f}  shipped={f:.4f}  Δ={f-b:+.4f}")

In [ ]:
for col, tag in [("baseline_256_score", "baseline"), ("ft_e2_score", "cartographer (shipped)")]:
    print(f"\n{tag} — mean score by true label:")
    g = df.groupby("esci_label")[col].agg(["mean", "std", "count"])
    print(g.reindex(["E", "S", "C", "I"]).round(4))

In [ ]:
print("test queries:", df["query_id"].nunique())
print("test pairs:", len(df))
n_params = sum(p.numel() for p in model_e2.model.parameters())
print("shipped model params:", f"{n_params/1e6:.1f}M")

In [ ]:
from sentence_transformers import CrossEncoder
check = CrossEncoder("2013khansohail/cartographer-ecommerce-reranker-MiniLM-L6-v2", max_length=256)
print(check.predict([("wireless headphones", "Sony WH-1000XM5 Wireless Noise Cancelling Headphones")]))